In [ ]:
# ============================================================
# Imports and helper functions
# ============================================================

from pathlib import Path

import numpy as np
import awkward as ak
import uproot
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

def flatten_finite(x):
    """
    Flatten an Awkward array and keep only finite values.
    """
    x = ak.to_numpy(ak.flatten(x, axis=None))
    return x[np.isfinite(x)]


def safe_percentile(values, percentile, fallback=1.0):
    """
    Percentile helper that does not crash on empty arrays.
    """
    values = np.asarray(values)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return fallback

    return np.percentile(values, percentile)


def step_hist(ax, values, bin_edges, *, color, label, linewidth=2.5, alpha=0.35):
    """
    Plot histogram as a step line using ax.plot(..., drawstyle='steps-post').
    """
    values = np.asarray(values)
    values = values[np.isfinite(values)]

    counts, _ = np.histogram(values, bins=bin_edges)

    ax.plot(
        bin_edges[:-1],
        counts,
        drawstyle="steps-post",
        color=color,
        linewidth=linewidth,
        alpha=alpha,
        label=label,
    )

    return counts


def step_hist_normalized(ax, values, bin_edges, *, color, label, linewidth=2.5, alpha=0.35):
    """
    Plot a normalized histogram as a step line.

    Normalization:
        counts / total_counts
    """
    values = np.asarray(values)
    values = values[np.isfinite(values)]

    counts, _ = np.histogram(values, bins=bin_edges)

    total = np.sum(counts)

    if total > 0:
        counts = counts / total

    ax.plot(
        bin_edges[:-1],
        counts,
        drawstyle="steps-post",
        color=color,
        linewidth=linewidth,
        alpha=alpha,
        label=label,
    )

    return counts

In [ ]:
# ============================================================
# Plot style settings
# ============================================================

FIGSIZE = (7.2, 5.6)

TITLE_FONTSIZE = 20
LABEL_FONTSIZE = 17
TICK_FONTSIZE = 15
LEGEND_FONTSIZE = 12

LINEWIDTH_MAIN = 2.5
LINEWIDTH_CAT = 2.
ALPHA_CAT = 0.85
ALPHA_MAIN = 0.95

TOTAL_COLOR = "#012020"


def style_axes(ax):
    ax.tick_params(
        axis="both",
        which="major",
        labelsize=TICK_FONTSIZE,
        width=1.3,
        length=6,
        direction="in",
    )

    ax.tick_params(
        axis="both",
        which="minor",
        width=1.0,
        length=3,
        direction="in",
    )

    ax.minorticks_on()

    for spine in ax.spines.values():
        spine.set_linewidth(1.3)

    ax.grid(True, which="major", alpha=0.25, linewidth=0.8)
    ax.grid(True, which="minor", alpha=0.12, linewidth=0.5)

In [ ]:
# ============================================================
# Read merged ROOT file and print summaries
# ============================================================

merged_root = Path("output/wcte_ambe_mc_plus_clean_bkg_pe.root")
#merged_root = Path("output/wcte_ambe_mc_plus_clean_bkg_pe_plus_59kevgamma.root")

# time_branch = "hit_pmt_times"
time_branch = "hit_pmt_calibrated_times"

# ------------------------------------------------------------
# Branch lists
# ------------------------------------------------------------

wcte_branches = [
    "window_time",
    "start_counter",
    "run_id",
    "sub_run_id",
    "spill_counter",
    "readout_number",

    "trigger_types",
    "trigger_times",

    "led_gains",
    "led_dacsettings",
    "led_ids",
    "led_card_ids",
    "led_slot_numbers",
    "led_event_types",
    "led_types",
    "led_sequence_numbers",
    "led_counters",

    "pmt_waveform_mpmt_card_ids",
    "pmt_waveform_pmt_channel_ids",
    "pmt_waveform_mpmt_slot_ids",
    "pmt_waveform_pmt_position_ids",
    "pmt_waveform_times",
    "pmt_waveforms",

    "beamline_pmt_qdc_charges",
    "beamline_pmt_tdc_times",
    "beamline_pmt_qdc_ids",
    "beamline_pmt_tdc_ids",

    "event_number",
    "hit_pmt_times",
    "hit_pmt_calibrated_times",
    "hit_pmt_charges",
    "hit_mpmt_slot_ids",
    "hit_pmt_position_ids",
    "hit_mpmt_card_ids",
    "hit_pmt_channel_ids",
    "hit_pmt_has_time_constant",
]

truth_branches = [
    "event_number",
    "n_overlay",

    "x",
    "y",
    "z",

    "dx",
    "dy",
    "dz",

    "hit_from_prompt",
    "hit_from_capture",
    "source_event_idx",

    "capture_t",
    "relative_capture_t",
    "prompt_time_withcapture",
    "prompt_time",

    "capture_x",
    "capture_y",
    "capture_z",

    "capture_nucleus",
    "capture_ngamma",
    "capture_total_gammaE",

    # Added during MC + background merge
    "is_background",
]

# ------------------------------------------------------------
# Read file
# ------------------------------------------------------------

with uproot.open(merged_root) as f:
    print("Top-level keys:")
    for key in f.keys():
        print(" ", key)

    wcte = f["WCTEReadoutWindows"]
    truth = f["TTrueInfo"]

    print()
    print("WCTE entries:", wcte.num_entries)
    print("TTrueInfo entries:", truth.num_entries)

    print()
    print("WCTE branches:")
    for key in wcte.keys():
        print(" ", key)

    print()
    print("TTrueInfo branches:")
    for key in truth.keys():
        print(" ", key)

    # Check that requested branches exist
    missing_wcte = [b for b in wcte_branches if b not in wcte.keys()]
    missing_truth = [b for b in truth_branches if b not in truth.keys()]

    if len(missing_wcte) > 0:
        raise RuntimeError(
            "Missing WCTEReadoutWindows branches:\n"
            + "\n".join(missing_wcte)
        )

    if len(missing_truth) > 0:
        raise RuntimeError(
            "Missing TTrueInfo branches:\n"
            + "\n".join(missing_truth)
        )

    # Read everything requested
    arrays = wcte.arrays(
        wcte_branches,
        library="ak",
    )

    truth_arrays = truth.arrays(
        truth_branches,
        library="ak",
    )

print()
print("Loaded WCTE branches:", len(arrays.fields))
print("Loaded TTrueInfo branches:", len(truth_arrays.fields))
print("Has TTrueInfo/is_background:", "is_background" in truth_arrays.fields)
print("Has TTrueInfo/relative_capture_t:", "relative_capture_t" in truth_arrays.fields)

# ------------------------------------------------------------
# Build category masks
# ------------------------------------------------------------

is_background = truth_arrays["is_background"]
is_mc = ~is_background

is_prompt = is_mc & (truth_arrays["hit_from_prompt"] == 1)
is_capture = is_mc & (truth_arrays["hit_from_capture"] == 1)
is_other_mc = is_mc & ~(is_prompt | is_capture)

# ------------------------------------------------------------
# Window-level counts
# ------------------------------------------------------------

n_hits_per_window = ak.to_numpy(
    ak.num(arrays[time_branch], axis=1)
)

n_label_per_window = ak.to_numpy(
    ak.num(is_background, axis=1)
)

n_prompt_per_window = ak.to_numpy(
    ak.sum(is_prompt, axis=1)
)

n_capture_per_window = ak.to_numpy(
    ak.sum(is_capture, axis=1)
)

n_other_mc_per_window = ak.to_numpy(
    ak.sum(is_other_mc, axis=1)
)

n_bkg_per_window = ak.to_numpy(
    ak.sum(is_background, axis=1)
)

n_mc_per_window = (
    n_prompt_per_window
    + n_capture_per_window
    + n_other_mc_per_window
)

window_sum_charges_pe = ak.to_numpy(
    ak.fill_none(
        ak.sum(arrays["hit_pmt_charges"], axis=1, mask_identity=True),
        0.0,
    )
)

finite_charge_mask = np.isfinite(window_sum_charges_pe)
window_sum_charges_pe_finite = window_sum_charges_pe[finite_charge_mask]

# ------------------------------------------------------------
# Capture-time summary
# ------------------------------------------------------------
relative_capture_t_all = ak.to_numpy(
    ak.flatten(truth_arrays["relative_capture_t"], axis=None)
)

relative_capture_t_all = np.asarray(relative_capture_t_all, dtype=float)
relative_capture_t_all = relative_capture_t_all[np.isfinite(relative_capture_t_all)]
relative_capture_t_all = relative_capture_t_all[relative_capture_t_all >= 0]

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print()
print("Hit-count alignment check:")
print(
    "  WCTE hit counts equal is_background counts:",
    np.array_equal(n_hits_per_window, n_label_per_window),
)

print()
print("Window summary:")
print("  Number of windows:", len(n_hits_per_window))
print("  Total hits:", int(np.sum(n_hits_per_window)))
print("  Total MC hits:", int(np.sum(n_mc_per_window)))
print("  Total prompt MC hits:", int(np.sum(n_prompt_per_window)))
print("  Total capture MC hits:", int(np.sum(n_capture_per_window)))
print("  Total other MC hits:", int(np.sum(n_other_mc_per_window)))
print("  Total background hits:", int(np.sum(n_bkg_per_window)))

print()
print("Hits per window:")
print("  Min:", int(np.min(n_hits_per_window)))
print("  Max:", int(np.max(n_hits_per_window)))
print("  Median:", np.median(n_hits_per_window))
print("  99th percentile:", np.percentile(n_hits_per_window, 99))

print()
print("Total charge per window [p.e.]:")
print("  Min:", np.min(window_sum_charges_pe_finite))
print("  Max:", np.max(window_sum_charges_pe_finite))
print("  Median:", np.median(window_sum_charges_pe_finite))
print("  99th percentile:", np.percentile(window_sum_charges_pe_finite, 99))

print()
print("Relative capture times:")
print("  Entries:", len(relative_capture_t_all))

if len(relative_capture_t_all) > 0:
    print("  Min [ns]:", np.min(relative_capture_t_all))
    print("  Max [ns]:", np.max(relative_capture_t_all))
    print("  Mean [ns]:", np.mean(relative_capture_t_all))
    print("  Median [ns]:", np.median(relative_capture_t_all))

print()
print("First 10 windows:")
for i in range(min(10, len(n_hits_per_window))):
    print(
        f"  window {i:4d}: "
        f"event={int(arrays['event_number'][i]):6d}, "
        f"total={int(n_hits_per_window[i]):6d}, "
        f"prompt={int(n_prompt_per_window[i]):5d}, "
        f"capture={int(n_capture_per_window[i]):5d}, "
        f"other_mc={int(n_other_mc_per_window[i]):5d}, "
        f"bkg={int(n_bkg_per_window[i]):6d}, "
        f"sumQ={window_sum_charges_pe[i]:10.3f} p.e."
    )

# Time and charge histograms

In [ ]:
# ------------------------------------------------------------
# Flatten hit-level quantities by category
# ------------------------------------------------------------

prompt_times = flatten_finite(arrays[time_branch][is_prompt])
capture_times = flatten_finite(arrays[time_branch][is_capture])
other_mc_times = flatten_finite(arrays[time_branch][is_other_mc])
bkg_times = flatten_finite(arrays[time_branch][is_background])

prompt_charges = flatten_finite(arrays["hit_pmt_charges"][is_prompt])
capture_charges = flatten_finite(arrays["hit_pmt_charges"][is_capture])
other_mc_charges = flatten_finite(arrays["hit_pmt_charges"][is_other_mc])
bkg_charges = flatten_finite(arrays["hit_pmt_charges"][is_background])
total_charges = flatten_finite(arrays["hit_pmt_charges"])

# ------------------------------------------------------------
# Window-level quantities by category
# ------------------------------------------------------------

n_prompt_hits_per_window = ak.to_numpy(
    ak.sum(is_prompt, axis=1)
)

n_capture_hits_per_window = ak.to_numpy(
    ak.sum(is_capture, axis=1)
)

n_other_mc_hits_per_window = ak.to_numpy(
    ak.sum(is_other_mc, axis=1)
)

n_bkg_hits_per_window = ak.to_numpy(
    ak.sum(is_background, axis=1)
)

n_total_hits_per_window = ak.to_numpy(
    ak.num(arrays[time_branch], axis=1)
)

prompt_sum_charges_per_window = ak.to_numpy(
    ak.fill_none(
        ak.sum(arrays["hit_pmt_charges"][is_prompt], axis=1, mask_identity=True),
        0.0,
    )
)

capture_sum_charges_per_window = ak.to_numpy(
    ak.fill_none(
        ak.sum(arrays["hit_pmt_charges"][is_capture], axis=1, mask_identity=True),
        0.0,
    )
)

other_mc_sum_charges_per_window = ak.to_numpy(
    ak.fill_none(
        ak.sum(arrays["hit_pmt_charges"][is_other_mc], axis=1, mask_identity=True),
        0.0,
    )
)

bkg_sum_charges_per_window = ak.to_numpy(
    ak.fill_none(
        ak.sum(arrays["hit_pmt_charges"][is_background], axis=1, mask_identity=True),
        0.0,
    )
)

total_sum_charges_per_window = ak.to_numpy(
    ak.fill_none(
        ak.sum(arrays["hit_pmt_charges"], axis=1, mask_identity=True),
        0.0,
    )
)

# Keep only finite charge values for plotting
prompt_sum_charges_per_window = prompt_sum_charges_per_window[
    np.isfinite(prompt_sum_charges_per_window)
]

capture_sum_charges_per_window = capture_sum_charges_per_window[
    np.isfinite(capture_sum_charges_per_window)
]

other_mc_sum_charges_per_window = other_mc_sum_charges_per_window[
    np.isfinite(other_mc_sum_charges_per_window)
]

bkg_sum_charges_per_window = bkg_sum_charges_per_window[
    np.isfinite(bkg_sum_charges_per_window)
]

total_sum_charges_per_window = total_sum_charges_per_window[
    np.isfinite(total_sum_charges_per_window)
]



# ============================================================
# 1) Hit time histogram by category, normalized
# ============================================================

time_bin_edges = np.linspace(0, 270_000, 13501)  # 20 ns bins

fig, ax = plt.subplots(figsize=FIGSIZE)

step_hist_normalized(
    ax,
    prompt_times,
    time_bin_edges,
    color="tab:blue",
    label=f"Prompt MC ({len(prompt_times)} hits)",
    linewidth=LINEWIDTH_CAT,
    alpha=ALPHA_CAT,
)

step_hist_normalized(
    ax,
    capture_times,
    time_bin_edges,
    color="tab:orange",
    label=f"Capture MC ({len(capture_times)} hits)",
    linewidth=LINEWIDTH_CAT,
    alpha=ALPHA_CAT,
)

step_hist_normalized(
    ax,
    other_mc_times,
    time_bin_edges,
    color="tab:red",
    label=f"Other MC ({len(other_mc_times)} hits)",
    linewidth=LINEWIDTH_CAT,
    alpha=ALPHA_CAT,
)

step_hist_normalized(
    ax,
    bkg_times,
    time_bin_edges,
    color="tab:green",
    label=f"Background ({len(bkg_times)} hits)",
    linewidth=LINEWIDTH_CAT,
    alpha=ALPHA_CAT,
)

ax.set_xlabel(f"{time_branch} [ns]", fontsize=LABEL_FONTSIZE)
ax.set_ylabel("Fraction of hits per bin", fontsize=LABEL_FONTSIZE)
ax.set_title("Normalized Hit Time Distribution by Category", fontsize=TITLE_FONTSIZE, pad=14)

style_axes(ax)

ax.legend(
    fontsize=LEGEND_FONTSIZE,
    frameon=True,
    framealpha=0.95,
    edgecolor="0.3",
)

plt.tight_layout()
plt.show()


# ============================================================
# 2) Per-hit charge histogram by category, normalized
# ============================================================

all_hit_charges = np.concatenate(
    [
        prompt_charges,
        capture_charges,
        other_mc_charges,
        bkg_charges,
    ]
)

charge_hit_xmax = 4 #safe_percentile(all_hit_charges, 99.5, fallback=10.0)
charge_hit_bin_edges = np.linspace(0, charge_hit_xmax, 101)

fig, ax = plt.subplots(figsize=FIGSIZE)

step_hist_normalized(
    ax,
    prompt_charges,
    charge_hit_bin_edges,
    color="tab:blue",
    label=f"Prompt MC ({len(prompt_charges)} hits)",
    linewidth=LINEWIDTH_CAT,
    alpha=ALPHA_CAT,
)

step_hist_normalized(
    ax,
    capture_charges,
    charge_hit_bin_edges,
    color="tab:orange",
    label=f"Capture MC ({len(capture_charges)} hits)",
    linewidth=LINEWIDTH_CAT,
    alpha=ALPHA_CAT,
)

step_hist_normalized(
    ax,
    other_mc_charges,
    charge_hit_bin_edges,
    color="tab:red",
    label=f"Other MC ({len(other_mc_charges)} hits)",
    linewidth=LINEWIDTH_CAT,
    alpha=ALPHA_CAT,
)

step_hist_normalized(
    ax,
    bkg_charges,
    charge_hit_bin_edges,
    color="tab:green",
    label=f"Background ({len(bkg_charges)} hits)",
    linewidth=LINEWIDTH_CAT,
    alpha=ALPHA_CAT,
)

step_hist_normalized(
    ax,
    total_charges,
    charge_hit_bin_edges,
    color=TOTAL_COLOR,
    label=f"Total ({len(total_charges)} hits)",
    linewidth=LINEWIDTH_MAIN,
    alpha=ALPHA_MAIN,
)

ax.set_xlabel("Hit charge [p.e.]", fontsize=LABEL_FONTSIZE)
ax.set_ylabel("Fraction of hits per bin", fontsize=LABEL_FONTSIZE)
ax.set_title("Normalized Hit Charge Distribution by Category", fontsize=TITLE_FONTSIZE, pad=14)

style_axes(ax)

ax.legend(
    fontsize=LEGEND_FONTSIZE,
    frameon=True,
    framealpha=0.95,
    edgecolor="0.3",
)

plt.tight_layout()
plt.show()


# ============================================================
# 3) Number of hits per window: total, prompt, capture, other MC, background
# ============================================================

hit_xmax = 1800 #safe_percentile(n_total_hits_per_window, 99.5, fallback=1.0)

hit_bin_edges = np.linspace(
    1,
    hit_xmax,
    201,
)

fig, ax = plt.subplots(figsize=FIGSIZE)


step_hist(
    ax,
    n_prompt_hits_per_window,
    hit_bin_edges,
    color="tab:blue",
    label="Prompt MC hits (> 0)",
    linewidth=LINEWIDTH_CAT,
    alpha=ALPHA_CAT,
)

step_hist(
    ax,
    n_capture_hits_per_window,
    hit_bin_edges,
    color="tab:orange",
    label="Capture MC hits (> 0)",
    linewidth=LINEWIDTH_CAT,
    alpha=ALPHA_CAT,
)

step_hist(
    ax,
    n_other_mc_hits_per_window,
    hit_bin_edges,
    color="tab:red",
    label="Other MC hits (> 0)",
    linewidth=LINEWIDTH_CAT,
    alpha=ALPHA_CAT,
)

step_hist(
    ax,
    n_bkg_hits_per_window,
    hit_bin_edges,
    color="tab:green",
    label="Background hits",
    linewidth=LINEWIDTH_CAT,
    alpha=ALPHA_CAT,
)

step_hist(
    ax,
    n_total_hits_per_window,
    hit_bin_edges,
    color=TOTAL_COLOR,
    label="Total hits",
    linewidth=LINEWIDTH_MAIN,
    alpha=ALPHA_MAIN,
)

ax.set_xlabel("Number of hits per window", fontsize=LABEL_FONTSIZE)
ax.set_ylabel("Number of windows", fontsize=LABEL_FONTSIZE)
ax.set_title("Hits per Merged Window by Category", fontsize=TITLE_FONTSIZE, pad=14)

style_axes(ax)

ax.legend(
    fontsize=LEGEND_FONTSIZE,
    frameon=True,
    framealpha=0.95,
    edgecolor="0.3",
)

plt.tight_layout()
plt.show()


# ============================================================
# 4) Total charge per window: total, prompt, capture, other MC, background
# ============================================================

charge_window_xmax = 2350 #safe_percentile(total_sum_charges_per_window, 99.5, fallback=1.0)

charge_window_bin_edges = np.linspace(
    1,
    charge_window_xmax,
    201,
)

fig, ax = plt.subplots(figsize=FIGSIZE)

step_hist(
    ax,
    prompt_sum_charges_per_window,
    charge_window_bin_edges,
    color="tab:blue",
    label="Prompt MC charge (> 0)",
    linewidth=LINEWIDTH_CAT,
    alpha=ALPHA_CAT,
)

step_hist(
    ax,
    capture_sum_charges_per_window,
    charge_window_bin_edges,
    color="tab:orange",
    label="Capture MC charge (> 0)",
    linewidth=LINEWIDTH_CAT,
    alpha=ALPHA_CAT,
)

step_hist(
    ax,
    other_mc_sum_charges_per_window,
    charge_window_bin_edges,
    color="tab:red",
    label="Other MC charge (> 0)",
    linewidth=LINEWIDTH_CAT,
    alpha=ALPHA_CAT,
)

step_hist(
    ax,
    bkg_sum_charges_per_window,
    charge_window_bin_edges,
    color="tab:green",
    label="Background charge",
    linewidth=LINEWIDTH_CAT,
    alpha=ALPHA_CAT,
)

step_hist(
    ax,
    total_sum_charges_per_window,
    charge_window_bin_edges,
    color=TOTAL_COLOR,
    label="Total charge",
    linewidth=LINEWIDTH_MAIN,
    alpha=ALPHA_MAIN,
)

ax.set_xlabel("Total charge per window [p.e.]", fontsize=LABEL_FONTSIZE)
ax.set_ylabel("Number of windows", fontsize=LABEL_FONTSIZE)
ax.set_title("Charge per Merged Window by Category", fontsize=TITLE_FONTSIZE, pad=14)

style_axes(ax)

ax.legend(
    fontsize=LEGEND_FONTSIZE,
    frameon=True,
    framealpha=0.95,
    edgecolor="0.3",
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# tRMS of hits split by prompt, capture, other MC, and background
# ============================================================

def trms_per_window(times_by_window):
    """
    Compute hit-time RMS per window.

    Returns NaN for windows with fewer than 2 valid hits.
    """
    out = []

    for times_evt in times_by_window:
        t_evt = np.asarray(times_evt, dtype=float)
        t_evt = t_evt[np.isfinite(t_evt)]

        if len(t_evt) >= 2:
            out.append(np.std(t_evt))
        else:
            out.append(np.nan)

    return np.asarray(out, dtype=float)


# ------------------------------------------------------------
# Compute tRMS per category
# ------------------------------------------------------------

prompt_trms = trms_per_window(
    arrays[time_branch][is_prompt]
)

capture_trms = trms_per_window(
    arrays[time_branch][is_capture]
)

other_mc_trms = trms_per_window(
    arrays[time_branch][is_other_mc]
)

bkg_trms = trms_per_window(
    arrays[time_branch][is_background]
)

total_trms = trms_per_window(
    arrays[time_branch]
)

# Keep finite values
prompt_trms_finite = prompt_trms[np.isfinite(prompt_trms)]
capture_trms_finite = capture_trms[np.isfinite(capture_trms)]
other_mc_trms_finite = other_mc_trms[np.isfinite(other_mc_trms)]
bkg_trms_finite = bkg_trms[np.isfinite(bkg_trms)]
total_trms_finite = total_trms[np.isfinite(total_trms)]

print("Finite tRMS entries:")
print("  Total:", len(total_trms_finite))
print("  Prompt MC:", len(prompt_trms_finite))
print("  Capture MC:", len(capture_trms_finite))
print("  Other MC:", len(other_mc_trms_finite))
print("  Background:", len(bkg_trms_finite))

print()
print("Median tRMS [ns]:")
print("  Total:", np.median(total_trms_finite))
print("  Prompt MC:", np.median(prompt_trms_finite) if len(prompt_trms_finite) > 0 else np.nan)
print("  Capture MC:", np.median(capture_trms_finite) if len(capture_trms_finite) > 0 else np.nan)
print("  Other MC:", np.median(other_mc_trms_finite) if len(other_mc_trms_finite) > 0 else np.nan)
print("  Background:", np.median(bkg_trms_finite) if len(bkg_trms_finite) > 0 else np.nan)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

all_trms_for_range = np.concatenate(
    [
        total_trms_finite,
        prompt_trms_finite,
        capture_trms_finite,
        other_mc_trms_finite,
        bkg_trms_finite,
    ]
)

trms_xmax = safe_percentile(
    all_trms_for_range,
    99.5,
    fallback=1.0,
)

trms_bin_edges = np.linspace(
    0,
    trms_xmax,
    201,
)

fig, ax = plt.subplots(figsize=(10, 6))

step_hist(
    ax,
    total_trms_finite,
    trms_bin_edges,
    color="black",
    label=f"Total ({len(total_trms_finite)} windows)",
    linewidth=2.8,
    alpha=0.95,
)

step_hist(
    ax,
    prompt_trms_finite,
    trms_bin_edges,
    color="tab:blue",
    label=f"Prompt MC ({len(prompt_trms_finite)} windows)",
    linewidth=2.4,
    alpha=0.55,
)

step_hist(
    ax,
    capture_trms_finite,
    trms_bin_edges,
    color="tab:orange",
    label=f"Capture MC ({len(capture_trms_finite)} windows)",
    linewidth=2.4,
    alpha=0.55,
)

step_hist(
    ax,
    other_mc_trms_finite,
    trms_bin_edges,
    color="tab:red",
    label=f"Other MC ({len(other_mc_trms_finite)} windows)",
    linewidth=2.4,
    alpha=0.55,
)

step_hist(
    ax,
    bkg_trms_finite,
    trms_bin_edges,
    color="tab:green",
    label=f"Background ({len(bkg_trms_finite)} windows)",
    linewidth=2.4,
    alpha=0.55,
)

ax.set_xlabel(f"tRMS of {time_branch} [ns]", fontsize=14)
ax.set_ylabel("Number of windows", fontsize=14)
ax.set_title("Hit-time RMS per window by category", fontsize=16)
ax.legend(fontsize=12)

plt.tight_layout()
plt.show()

# ============================================================
# tRMS of MC hits only: prompt, capture, and other MC
# ============================================================

# Use only the MC categories, no total and no background
mc_trms_for_range = np.concatenate(
    [
        prompt_trms_finite,
        capture_trms_finite,
        other_mc_trms_finite,
    ]
)


trms_bin_edges_mc = np.linspace(
    -1,
    400,
    402,
)

fig, ax = plt.subplots(figsize=(10, 6))

step_hist(
    ax,
    prompt_trms_finite,
    trms_bin_edges_mc,
    color="tab:blue",
    label=f"Prompt MC ({len(prompt_trms_finite)} windows)",
    linewidth=2.8,
    alpha=0.85,
)

step_hist(
    ax,
    capture_trms_finite,
    trms_bin_edges_mc,
    color="tab:orange",
    label=f"Capture MC ({len(capture_trms_finite)} windows)",
    linewidth=2.8,
    alpha=0.85,
)

step_hist(
    ax,
    other_mc_trms_finite,
    trms_bin_edges_mc,
    color="tab:red",
    label=f"Other MC ({len(other_mc_trms_finite)} windows)",
    linewidth=2.8,
    alpha=0.85,
)

ax.set_yscale("log")
ax.set_xlabel(f"tRMS of {time_branch} [ns]", fontsize=14)
ax.set_ylabel("Number of windows", fontsize=14)
ax.set_title("Hit-time RMS per window: prompt, capture, and other MC", fontsize=16)
ax.legend(fontsize=12)

plt.tight_layout()
plt.show()

# Capture time and exponential fit

In [ ]:
# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------

t = np.asarray(relative_capture_t_all, dtype=float)
t = t[np.isfinite(t)]
t = t[t >= 0]

if len(t) == 0:
    raise RuntimeError("No valid relative capture times found.")

# ------------------------------------------------------------
# Summary statistics
# ------------------------------------------------------------

mean_t = np.mean(t)

print("Relative capture time entries:", len(t))
print(f"Mean relative capture time = {mean_t:.3f} ns")

# ------------------------------------------------------------
# Histogram
# ------------------------------------------------------------

nbins = 100

counts, bin_edges = np.histogram(
    t,
    bins=nbins,
)

bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# ------------------------------------------------------------
# Exponential fit
# ------------------------------------------------------------

def expo_model(x, A, tau):
    return A * np.exp(-x / tau)

A0 = counts.max()
tau0 = mean_t if mean_t > 0 else 1.0

fit_mask = counts > 0

# Skip first bin
if len(fit_mask) > 0:
    fit_mask[0] = False

if np.count_nonzero(fit_mask) < 2:
    raise RuntimeError("Not enough nonzero bins to fit an exponential.")

popt, pcov = curve_fit(
    expo_model,
    bin_centers[fit_mask],
    counts[fit_mask],
    p0=[A0, tau0],
    maxfev=10000,
)

A_fit, tau_fit = popt
tau_err = np.sqrt(np.diag(pcov))[1]

print(f"Fitted tau = {tau_fit:.3f} ± {tau_err:.3f} ns")

# ------------------------------------------------------------
# Smooth curve for plotting
# ------------------------------------------------------------

xfit = np.linspace(
    bin_edges[1],
    bin_edges[-1],
    500,
)

yfit = expo_model(
    xfit,
    A_fit,
    tau_fit,
)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 6))

ax.hist(
    t,
    bins=nbins,
    color="teal",
    alpha=0.85,
    edgecolor="black",
    linewidth=0.8,
    label="Relative capture times",
)

ax.plot(
    xfit,
    yfit,
    color="black",
    linewidth=2.2,
    label=(
        fr"Exponential fit: $\tau = {tau_fit:.2f} \pm {tau_err:.2f}$ ns"
        "\n"
        fr"Mean = {mean_t:.2f} ns"
    ),
)

ax.set_xlabel("Relative capture time [ns]", fontsize=16)
ax.set_ylabel("Counts", fontsize=16)
ax.set_title("Distribution of relative capture times", fontsize=18)
ax.tick_params(axis="both", labelsize=13)
ax.legend(fontsize=12)

plt.tight_layout()
plt.show()

# Example window visualization.

In [ ]:
# ============================================================
# Example window containing prompt + capture + background hits
# ============================================================

# Change this if you want another candidate:
for candidate_rank in range(20):
    candidate_mask = (
        ((n_prompt_per_window > 0)
        | (n_capture_per_window > 0))
        & (n_bkg_per_window > 0)
    )

    candidate_indices = np.where(candidate_mask)[0]

    if len(candidate_indices) == 0:
        raise RuntimeError(
            "No window found with prompt hits, capture hits, and background hits."
        )

    if candidate_rank >= len(candidate_indices):
        raise RuntimeError(
            f"candidate_rank={candidate_rank} is too large. "
            f"There are only {len(candidate_indices)} candidate windows."
        )

    window_idx = int(candidate_indices[candidate_rank])

    print("Selected window index:", window_idx)
    print("Event number:", int(arrays["event_number"][window_idx]))
    print("Prompt hits:", int(n_prompt_per_window[window_idx]))
    print("Capture hits:", int(n_capture_per_window[window_idx]))
    print("Other MC hits:", int(n_other_mc_per_window[window_idx]))
    print("Background hits:", int(n_bkg_per_window[window_idx]))
    print("Total MC hits:", int(n_mc_per_window[window_idx]))
    print("Total hits:", int(n_hits_per_window[window_idx]))
    print("Total charge [p.e.]:", float(window_sum_charges_pe[window_idx]))

    # ------------------------------------------------------------
    # Extract selected-window times
    # ------------------------------------------------------------

    times_this_window = arrays[time_branch][window_idx]

    prompt_times_w = ak.to_numpy(times_this_window[is_prompt[window_idx]])
    capture_times_w = ak.to_numpy(times_this_window[is_capture[window_idx]])
    other_mc_times_w = ak.to_numpy(times_this_window[is_other_mc[window_idx]])
    bkg_times_w = ak.to_numpy(times_this_window[is_background[window_idx]])

    prompt_times_w = prompt_times_w[np.isfinite(prompt_times_w)]
    capture_times_w = capture_times_w[np.isfinite(capture_times_w)]
    other_mc_times_w = other_mc_times_w[np.isfinite(other_mc_times_w)]
    bkg_times_w = bkg_times_w[np.isfinite(bkg_times_w)]

    # ------------------------------------------------------------
    # Plot selected window
    # ------------------------------------------------------------

    bin_edges = np.linspace(0, 270_000, 13501)  # 20 ns bins

    fig, ax = plt.subplots(figsize=(10, 6))

    step_hist(
        ax,
        prompt_times_w,
        bin_edges,
        color="tab:blue",
        label=f"Prompt MC ({len(prompt_times_w)} hits)",
        alpha=0.75,
    )

    step_hist(
        ax,
        capture_times_w,
        bin_edges,
        color="tab:orange",
        label=f"Capture MC ({len(capture_times_w)} hits)",
        alpha=0.75,
    )

    step_hist(
        ax,
        other_mc_times_w,
        bin_edges,
        color="tab:red",
        label=f"Other MC ({len(other_mc_times_w)} hits)",
        alpha=0.75,
    )
    

    step_hist(
        ax,
        bkg_times_w,
        bin_edges,
        color="tab:green",
        label=f"Background ({len(bkg_times_w)} hits)",
        alpha=0.75,
    )

    ax.set_xlabel(f"{time_branch} [ns]", fontsize=14)
    ax.set_ylabel("Hits per bin", fontsize=14)
    ax.set_title(
        f"Example window {window_idx} | event_number={int(arrays['event_number'][window_idx])}",
        fontsize=16,
    )
    ax.legend(fontsize=12)

    plt.tight_layout()
    plt.show()